[![Open In Colab](https://colab.research.google.com/assets/colab-badge.svg)](https://colab.research.google.com/github/petro1eum/trust_chain/blob/main/examples/trustchain_receipts.ipynb)

# 🧾 `.tcreceipt` — Portable Proof of Every Signed Fact

> **Signature is in the package, not in the server.**

TrustChain signs every fact your AI pulls from a real tool. A `.tcreceipt`
is how you *hand that proof to someone else*.

It is one JSON file that bundles:

| Field        | What it carries                                                              |
|--------------|------------------------------------------------------------------------------|
| `envelope`   | The exact bytes that were signed (`tool_id`, `data`, `timestamp`, `nonce`…)  |
| `signature`  | The Ed25519 signature                                                        |
| `key`        | Public key (+ optional `key_id`) the recipient needs to verify               |
| `identity`   | *(optional)* X.509 chain, subject CN — who signed it                         |
| `witnesses`  | *(optional)* TSA / third-party co-signers                                    |

Three verifiers are **byte-for-byte compatible**:

1. **Python** — `trustchain.Receipt.verify()` / `tc receipt verify` CLI.
2. **JS SDK** — `trustchain/js_sdk/receipt.mjs` (Node 18+, browser bundlers).
3. **Pure browser** — `examples/verify.html`, zero deps, WebCrypto, works offline.

In this notebook we cover the four workflows you actually need in production:

1. **Build + download** a receipt from a signed fact.
2. **Verify** it end-to-end, with pinning and max-age.
3. **Detect tampering** — what exit codes / errors you get.
4. **Identity-bound** receipts for compliance (X.509 attached).

> Run `!pip install trustchain` in the first cell if you're on Colab.

In [ ]:
!pip install -q trustchain

## 1️⃣ Build a receipt from a signed fact

`tc.sign(tool_id, data)` returns a `SignedResponse`. Pass it to `build_receipt()`
together with your public key (and optional `key_id`). The result serializes to
a single JSON blob — save it with the `.tcreceipt` extension for convention.

In [ ]:
from pathlib import Path

from trustchain import Receipt, TrustChain, TrustChainConfig, build_receipt

tc = TrustChain(TrustChainConfig(enable_nonce=False))

# Pretend an AI tool just returned a fact we care about.
signed = tc.sign(
    tool_id="sec_filing_lookup",
    data={
        "company": "Acme Corp",
        "filing": "10-K",
        "fiscal_year": 2025,
        "revenue_usd": 4_812_300_000,
    },
)
assert tc.verify(signed), "freshly signed response must verify"

receipt = build_receipt(
    signed,
    public_key_b64=tc.export_public_key(),
    key_id=tc.get_key_id(),  # lets downstream systems pin by key id
)

out = Path("acme_10k.tcreceipt")
out.write_text(receipt.to_json(indent=2))

print(f"💾 wrote {out}  ({out.stat().st_size} bytes)")
print("\n--- first 320 chars ---")
print(out.read_text()[:320], "…")

## 2️⃣ Verify a receipt — with pinning and freshness

`Receipt.verify()` runs the **canonical envelope reconstruction** + Ed25519
check. Optional knobs that matter in real deployments:

* `expected_public_key_b64` — *pin* the receipt to a known identity.
  Without this, you're only verifying internal consistency.
* `max_age_seconds` — reject stale receipts (login challenges, one-shot
  tool results).

The returned `ReceiptVerification` gives you granular flags — not just a
boolean — so you can tell **why** something is invalid.

In [ ]:
loaded = Receipt.load(out.read_text())

v = loaded.verify(
    expected_public_key_b64=tc.export_public_key(),
    max_age_seconds=3600,
)

print("✅ valid       :", v.valid)
print("   signature_ok:", v.signature_ok)
print("   identity_ok :", v.identity_ok)
print("   witnesses_ok:", v.witnesses_ok)
print("   errors      :", v.errors or "—")
print("   warnings    :", v.warnings or "—")

## 3️⃣ Tamper detection — what you get when things go wrong

The whole point is that *any* modification breaks the signature.
Let's confirm, and see the exact error message you'd show to a user.

In [ ]:
import json

tampered = json.loads(out.read_text())
tampered["envelope"]["data"]["revenue_usd"] = 9_999_999_999  # 💰 lie

bad = Receipt._from_dict(tampered)
v_bad = bad.verify(expected_public_key_b64=tc.export_public_key())

print("❌ valid       :", v_bad.valid)
print("   signature_ok:", v_bad.signature_ok)
print("   errors      :", v_bad.errors)

# Same story if somebody swaps the public key to one they control —
# pinning is what stops this class of attacks.
wrong_key = TrustChain(TrustChainConfig(enable_nonce=False))
forged = loaded.verify(expected_public_key_b64=wrong_key.export_public_key())
print("\n🚫 pinning to wrong key → valid:", forged.valid, "| errors:", forged.errors)

### CLI equivalent — same exit codes, scriptable

The Python CLI mirrors this exactly — perfect for CI/CD gates, compliance
cron jobs, or webhook handlers.

```bash
tc receipt verify acme_10k.tcreceipt        # 0 = ok
tc receipt verify bad.tcreceipt             # 2 = tampered
tc receipt verify old.tcreceipt --max-age 60  # 3 = degraded (too old)
tc receipt verify garbage.tcreceipt         # 4 = malformed
```

## 4️⃣ Identity-bound receipts (compliance mode)

For regulated industries — HIPAA, SOC2, FDA — a raw public key is not enough.
You need **who** signed it, backed by an X.509 chain rooted in a CA they trust.

TrustChain lets you attach identity material that is covered *by the Ed25519
signature itself*, so the signer can't later repudiate "those aren't my certs".

In [ ]:
# Toy certificate for the demo — in production this comes from your internal CA.
demo_cert = {
    "subject_cn": "agent-prod-42.acme.example",
    "issuer_cn": "Acme Internal CA G2",
    "not_before": "2026-01-01T00:00:00Z",
    "not_after": "2027-01-01T00:00:00Z",
    "cert_chain_pem": ["-----BEGIN CERTIFICATE-----\n…\n-----END CERTIFICATE-----"],
}

tc_regulated = TrustChain(
    TrustChainConfig(enable_nonce=False, certificate=demo_cert),
)

signed_r = tc_regulated.sign(
    "diagnosis_lookup",
    {"patient_id": "P-99123", "icd10": "E11.9", "confidence": 0.94},
)

# build_receipt pulls identity from the signed response automatically —
# or pass it explicitly via `identity=...` if you need a different view.
rcpt_r = build_receipt(
    signed_r,
    public_key_b64=tc_regulated.export_public_key(),
    key_id=tc_regulated.get_key_id(),
    identity=demo_cert,
)

v_r = Receipt.load(rcpt_r.to_json()).verify(
    expected_public_key_b64=tc_regulated.export_public_key(),
)

print("👤 subject_cn :", rcpt_r.identity["subject_cn"])
print("🏛 issuer_cn  :", rcpt_r.identity["issuer_cn"])
print("✅ valid       :", v_r.valid)
print("   identity_ok:", v_r.identity_ok)

## 5️⃣ Verify in the browser — airgapped, no Python, no network

The `.tcreceipt` you wrote above is not Python-specific. Open
[`examples/verify.html`](https://github.com/petro1eum/trust_chain/blob/main/examples/verify.html)
on a laptop with no internet connection, drag `acme_10k.tcreceipt` onto it,
and you'll get the **same verdict** — WebCrypto Ed25519, canonicalization
identical to Python down to the byte.

This is the whole pitch: a regulator / partner / auditor *never needs to trust
your server*. They only need to trust the signer's public key.

### Integrating in a React app

```jsx
import { TrustReceipt } from 'trustchain/react';

<TrustReceipt file={uploadedFile}>
  {({ valid, envelope, errors }) =>
    valid
      ? <Badge ok>Verified · {envelope.tool_id}</Badge>
      : <Badge error>Invalid · {errors.join(', ')}</Badge>
  }
</TrustReceipt>
```

## 📚 TL;DR

| Use case                                     | API                                            |
|----------------------------------------------|------------------------------------------------|
| Create portable proof                        | `build_receipt(signed, pk, key_id=…)`          |
| Load + verify                                | `Receipt.load(json).verify(...)`               |
| Pin to known identity                        | `verify(expected_public_key_b64=…)`            |
| Reject stale                                 | `verify(max_age_seconds=…)`                    |
| CI / compliance gate                         | `tc receipt verify file.tcreceipt`             |
| Browser-side check                           | `examples/verify.html` — drag & drop           |
| React component                              | `<TrustReceipt/>` from `trustchain/js_sdk/react.jsx` |

**Next steps**

- `trustchain_tutorial.ipynb` — TrustChain fundamentals (Ed25519, chain of trust, tenants).
- `trustchain_llm.ipynb` — wrap OpenAI / Claude tool-calls with verified signatures.
- [GitHub wiki](https://github.com/petro1eum/trust_chain/wiki) — full API reference.
